# 00 · Setup & smoke test

**Purpose.** Verify the Colab environment, install dependencies, and run the three agents end-to-end on one image before committing GPU hours. Nothing here is used as a result.

| | |
|---|---|
| **Proposal objective** | Prerequisite for all objectives |
| **Inputs** | one COCO image downloaded from the web |
| **Outputs** | none (printouts + figures only) |
| **Runtime (T4)** | ~10 min (model downloads) |

> Run cells top to bottom. Every expensive stage is **resumable**: if Colab disconnects, reconnect,
> re-run from the top and it continues where it stopped (results are cached on Drive under `results/`).

## 1. Environment

In [ ]:
# ===================== Bootstrap (identical in every notebook) =====================
import os, sys, subprocess, shutil  # paths, sys.path, shell commands (git clone, pip), folder copy
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = (not IN_COLAB) and os.path.isdir("/kaggle/working")    # Kaggle notebook (needs Internet ON, see docs/KAGGLE_GUIDE.md)

# Where the project lives on Google Drive. Upload the whole repo folder there (see docs/COLAB_GUIDE.md),
# or set REPO_URL and it will be cloned into DRIVE_PROJECT on first run.
DRIVE_PROJECT = "/content/drive/MyDrive/detect-verify-decide"
REPO_URL = ""  # e.g. "https://github.com/<you>/detect-verify-decide.git"
BRANCH = "kaggle-version"  # git branch to take the latest code from (change it if you push to another branch)

if IN_COLAB:
    from google.colab import drive  # Colab only: mount Google Drive
    drive.mount("/content/drive")
    if not os.path.isdir(DRIVE_PROJECT) and REPO_URL:
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, DRIVE_PROJECT], check=True)
    elif os.path.isdir(os.path.join(DRIVE_PROJECT, ".git")):   # pick up the latest pushed code (never touches your notebooks/results)
        subprocess.run(["git", "-C", DRIVE_PROJECT, "fetch", "origin", BRANCH], check=False)
        subprocess.run(["git", "-C", DRIVE_PROJECT, "checkout", "FETCH_HEAD", "--", "src", "configs", "requirements.txt", "tools", "app"], check=False)
    PROJECT_ROOT = DRIVE_PROJECT
elif IN_KAGGLE:
    PROJECT_ROOT = "/kaggle/working/detect-verify-decide"
    if not os.path.isdir(os.path.join(PROJECT_ROOT, "src", "dvd")):
        import glob
        attached = glob.glob("/kaggle/input/*/src/dvd") + glob.glob("/kaggle/input/*/*/src/dvd")   # project attached as a Kaggle Dataset
        if attached:
            shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), PROJECT_ROOT)
        elif REPO_URL:
            url = REPO_URL
            try:   # private repo: add a Kaggle secret named GITHUB_TOKEN (Add-ons -> Secrets)
                from kaggle_secrets import UserSecretsClient
                url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
            except Exception:
                pass
            r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, PROJECT_ROOT], capture_output=True)
            if r.returncode:   # (the message deliberately does not echo the command, which could contain the token)
                raise RuntimeError("git clone failed: is Internet ON, REPO_URL / BRANCH right, and for a private repo the GITHUB_TOKEN secret set?")
            subprocess.run(["git", "-C", PROJECT_ROOT, "remote", "set-url", "origin", REPO_URL])   # never keep the token in .git/config
        else:
            raise RuntimeError("Set REPO_URL (and BRANCH) above, or attach the project as a Kaggle Dataset.")
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

assert os.path.isdir(os.path.join(PROJECT_ROOT, "src", "dvd")), f"src/dvd not found under {PROJECT_ROOT}"
os.environ["DVD_ROOT"] = PROJECT_ROOT
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
# Install dependencies (Colab / Kaggle already have torch/torchvision with CUDA). ~1-2 min per fresh runtime.
if IN_COLAB or IN_KAGGLE:
    %pip install -q -r "{PROJECT_ROOT}/requirements.txt"

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt  # json files, numpy arrays, pandas tables, matplotlib plots
from dvd.config import load_config, save_selected, results_dir  # load configs/default.yaml, save tuned values, results folders
from dvd.io_utils import set_seed, save_json, load_json, JsonlCache  # seeding, JSON helpers and the resumable JSONL cache
from dvd import paths  # cache-file path helpers

cfg = load_config()
set_seed(cfg["seed"])
pd.set_option("display.precision", 4)
try:
    import torch  # PyTorch: GPU check and memory statistics
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only - expect very slow runs)")
except ImportError:
    print("torch not installed")
print("data_root   :", cfg["paths"]["data_root"])
print("results_root:", cfg["paths"]["results_root"])
print("tuned params:", cfg["selected"] or "(none yet)")

In [ ]:
!nvidia-smi
import transformers, PIL  # version printout of the two core libraries
print("transformers", transformers.__version__, "| torch", torch.__version__, "| PIL", PIL.__version__)

## 2. A test image
COCO `000000039769` (two cats on a couch) is fetched directly; no dataset download needed yet.

In [ ]:
import io, urllib.request  # download the test image straight into memory
from PIL import Image  # Pillow image type
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(io.BytesIO(urllib.request.urlopen(url).read())).convert("RGB")
plt.imshow(image); plt.axis("off"); plt.show()

## 3. Detector Agent
A small vocabulary is used here; the real runs use all 80 COCO classes.

In [ ]:
from dvd.agents.detector import build_detector  # Detector Agent (Grounding DINO): factory
from dvd.viz import show_detections  # plotting helpers (detections overlay, PR frontier, crop gallery)
vocab = ["cat", "dog", "couch", "remote", "bed", "person", "tv"]
detector = build_detector(cfg, vocab, {n: k for k, n in enumerate(vocab)})
det = detector.detect(image)
print(f"{len(det['scores'])} candidates in {det['det_time_s']:.2f}s using {det['det_invocations']} prompt(s)")
show_detections(image, det["boxes"], det["labels"], det["scores"], title="Detector candidates"); plt.show()

## 4. Verifier Agent (crop-and-zoom, label-blind to confidence)

In [ ]:
from dvd.agents.verifier import build_verifier, make_views  # Verifier Agent (frozen Qwen2-VL): factory and crop builder
verifier = build_verifier(cfg)
views = make_views(image, det["boxes"], cfg, "crop")
out = verifier.verify(views, det["labels"])
pd.DataFrame({"label": det["labels"], "det_conf": det["scores"], "verifier_p_yes": out["p_yes"],
              "verdict": out["verdict"]})

In [ ]:
fig, axes = plt.subplots(1, len(views), figsize=(3 * len(views), 3), squeeze=False)
for ax, v, l, p in zip(axes[0], views, det["labels"], out["p_yes"]):
    ax.imshow(v); ax.set_title(f"{l}  P(yes)={p:.2f}", fontsize=9); ax.axis("off")
plt.show()

## 5. Hallucination probe
Ask about objects that are **not** in the image. A well-behaved verifier should give low P(yes).

In [ ]:
from dvd.cropping import fit_max_side  # crop-and-zoom / image resizing helpers
full = fit_max_side(image, cfg["verifier"]["full_image_max_side"])
qs = [f"Is there a {o} in the image? Answer yes or no." for o in ["cat", "remote", "dog", "car", "pizza"]]
for q, p in zip(qs, verifier.yes_no([full] * len(qs), qs)):
    print(f"{p:.3f}  {q}")

## 6. Full online pipeline (Detect -> Verify uncertain band -> Decide)

In [ ]:
from dvd.agents.decision import RuleDecisionAgent  # Decision Agent: the accept/reject rule and the optional LLM aggregator
from dvd.pipeline import DetectVerifyDecide  # cached offline stages and the online Detect-Verify-Decide pipeline
pipe = DetectVerifyDecide(detector, verifier, RuleDecisionAgent(tau_low=0.25, tau_high=0.6), cfg)
res = pipe(image)
status = ["accepted" if a else "rejected" for a in res["accept"]]
print(f"detector {res['det_time_s']:.2f}s | verifier {res['verify_time_s']:.2f}s on {res['verify_invocations']} crops | total {res['total_time_s']:.2f}s")
show_detections(image, res["boxes"], res["labels"], res["scores"], status, title="green = accepted, red dashed = rejected"); plt.show()
if torch.cuda.is_available():
    print(f"peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

## ✅ Smoke-test checklist
- [ ] GPU detected (T4 or better)
- [ ] Detector finds the two cats / couch / remotes
- [ ] Verifier P(yes) is high for true crops and low for absent objects in the probe
- [ ] Peak GPU memory comfortably below the card limit (both models fit together)

If all pass, continue with **01_data_preparation**.